In [ ]:

from pathlib import Path
import re

import pandas as pd

RAW_DATA_PATHS = [Path("Data/Raw data.csv"), Path("data/Raw data.csv")]
RAW_ENCODINGS = ["utf-8-sig", "utf-8", "gb18030", "cp1252", "latin1"]


def read_raw_data(paths=RAW_DATA_PATHS, encodings=RAW_ENCODINGS):
    """Load the anonymized CBE/Qualtrics export across OS path and encoding variants."""
    errors = []
    for file_path in paths:
        if not file_path.exists():
            errors.append(f"{file_path}: file not found")
            continue
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding)
                print(f"Loaded {file_path} using encoding={encoding}; shape={df.shape}")
                return df, file_path, encoding
            except UnicodeDecodeError as exc:
                errors.append(f"{file_path} with {encoding}: {exc}")
            except pd.errors.ParserError as exc:
                errors.append(f"{file_path} with {encoding}: {exc}")
    raise RuntimeError("Unable to load raw data. Tried:\n" + "\n".join(errors))


data, raw_data_path, raw_data_encoding = read_raw_data()
data.head()


In [ ]:

data.columns


In [ ]:

# Preprocess column names and values from the anonymized Qualtrics export.
def extract_column_name(column_name):
    """Extract the Qualtrics export tag from 'Question text (tag)' column names."""
    if not isinstance(column_name, str) or not column_name.endswith(")"):
        return column_name

    # Search from the right and require balanced parentheses so nested tags such as
    # 'Duration (in seconds) (Duration (in seconds))' become 'Duration (in seconds)'.
    for match in reversed(list(re.finditer(r" \(", column_name))):
        candidate = column_name[match.start() + 2:-1]
        if candidate.count("(") == candidate.count(")"):
            return candidate
    return column_name


def normalize_cell_value(value):
    """Normalize known encoding/Excel artifacts without changing valid survey labels."""
    if pd.isna(value):
        return pd.NA
    if not isinstance(value, str):
        return value

    value = value.strip()
    if value == "":
        return pd.NA

    replacements = {
        "11月20日": "11-20",
        "11ÔÂ20ÈÕ": "11-20",
        "11��20��": "11-20",
        "11–20": "11-20",
        "11—20": "11-20",
    }
    return replacements.get(value, value)


data.columns = [extract_column_name(name) for name in data.columns]
data = data.apply(lambda column: column.map(normalize_cell_value))

print(f"Rows: {data.shape[0]}, columns: {data.shape[1]}")
print(data.columns.tolist())
data.head()


In [ ]:
dictionary = {
 'Finished': {True: 1, False: 0, 'True': 1, 'False': 0, 'TRUE': 1, 'FALSE': 0, 1: 1, 0: 0},
 'background_time': {'Less than 3 months': 0, '3-6 months': 1, '7-12 months': 2, '1-2 years': 3, '2-3 years': 4, 'More than 3 years': 5},

 'background_hours': {'10 or less': 0, '11-20': 1, '21-30': 2, '31-40': 3, '41-50': 4 ,'More than 50': 5}, 

 'background_work': {'Staff': 5,  'Executive': 2,  'Management': 3,  'Other': 4,  'Administrative support': 0,  'C-level executive': 1},

 'background_gender': {'Female': 0, 'Male': 1},

 'iaq_sat_1': { 'Very dissatisfied': 5, 'Somewhat dissatisfied': 3, 'Dissatisfied': 0,  'Satisfied': 2,  'Very satisfied': 6,  'Somewhat satisfied': 4,  'Neither satisfied nor dissatisfied': 1,  }, 

 'iaq_dis_level_1': {'Not a problem': 0,  'Minor problem': 1,  'A problem': 2,  'Major problem': 3}, 
 'iaq_dis_level_2': {'Not a problem': 0,  'Minor problem': 1,  'A problem': 2,  'Major problem': 3}, 
 'iaq_dis_level_3': {'Not a problem': 0,  'Minor problem': 1,  'A problem': 2,  'Major problem': 3}, 

 'thermal_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6}, 
 'thermal_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'thermal_weather_hot': {'Be cooler': 0, 'Not change': 1, 'Be warmer': 2},
 'thermal_weather_cold': {'Be cooler': 2, 'Not change': 1, 'Be warmer': 0},


 'light_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'light_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'light_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'light_sat_4': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'light_task': {'Yes': 2, 'Not sure': 1, 'No': 0},

 'acoustic_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'acoustic_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'furnish_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'furnish_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'furnish_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},



 'layout_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'layout_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'layout_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'clean_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'clean_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},

 'amenities_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_sat_4': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_sat_5': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'location_near_1': {'No': 0, 'Yes': 1},
 'location_near_2': {'Yes': 1, 'No': 0},


 'amenities_prox_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_prox_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_prox_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_prox_sat_4': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'amenities_prox_sat_5': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Not applicable': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'water_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'water_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'water_dis_water': {'I dislike the taste': 0, 'Other': 1},
 'nutrition_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},

 'wellness_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'wellness_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'wellness_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'wellness_sat_4': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'wellness_sat_5': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'wellness_sat_6': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'My workplace does not have this policy': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},


 'health_rate_1': {'Extremely healthy': 0,  'Healthy': 1,  'Neither healthy nor unhealthy': 2,  'Unhealthy': 3,  'Extremely unhealthy': 4},
 'health_rate_2': {'Extremely healthy': 0,  'Healthy': 1,  'Neither healthy nor unhealthy': 2,  'Unhealthy': 3,  'Extremely unhealthy': 4},
 'health_mental_1': {'Extremely healthy': 0,  'Healthy': 1,  'Neither healthy nor unhealthy': 2,  'Unhealthy': 3,  'Extremely unhealthy': 4},
 'health_mental_2': {'Extremely healthy': 0,  'Healthy': 1,  'Neither healthy nor unhealthy': 2,  'Unhealthy': 3,  'Extremely unhealthy': 4},

 'health_interfere_1': {'Significantly enhances': 0,  'Enhances': 1,  'Somewhat enhances': 2,  'Neither enhances nor interferes': 3,  'Somewhat interferes': 4,  'Interferes': 5},
 'health_interfere_2': {'Significantly enhances': 0,  'Enhances': 1,  'Somewhat enhances': 2,  'Neither enhances nor interferes': 3,  'Somewhat interferes': 4,  'Interferes': 5},
 'health_interfere_3': {'Significantly enhances': 0,  'Enhances': 1,  'Somewhat enhances': 2,  'Neither enhances nor interferes': 3,  'Somewhat interferes': 4,  'Interferes': 5},
 'health_interfere_4': {'Significantly enhances': 0,  'Enhances': 1,  'Somewhat enhances': 2,  'Neither enhances nor interferes': 3,  'Somewhat interferes': 4,  'Interferes': 5},


 'activity_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},

 'support_sat_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'support_sat_2': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'support_sat_3': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},

 'support_life_1': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_life_2': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_life_3': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_life_4': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_life_5': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_life_6': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_job_1': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_job_2': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_job_3': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_job_4': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'support_job_5': {'Strongly agree': 0,  'Agree': 1,  'Slightly agree': 2, 'Neither agree nor disagree': 3, 'Slightly disagree': 4, 'Disagree': 5, 'Strongly disagree': 6},
 'general_enhance_1': {'Significantly enhances': 0,  'Enhances': 1,  'Somewhat enhances': 2,  'Neither enhances nor interferes': 3,  'Somewhat interferes': 4,  'Interferes': 5},

 'general_sat_space_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6},
 'general_sat_building_1': {'Very dissatisfied': 0, 'Dissatisfied': 1,  'Somewhat dissatisfied': 2, 'Neither satisfied nor dissatisfied': 3, 'Somewhat satisfied': 4, 'Satisfied': 5, 'Very satisfied': 6}
}


In [ ]:

# Apply explicit ordinal encodings for variables used in the analysis.
missing_dictionary_columns = []
unmapped_values = {}

for col, mapping in dictionary.items():
    if col not in data.columns:
        missing_dictionary_columns.append(col)
        continue

    original = data[col].copy()
    encoded = original.map(mapping)
    unknown = sorted(original[original.notna() & encoded.isna()].astype(str).unique())
    if unknown:
        unmapped_values[col] = unknown
    data[col] = encoded

print(f"Dictionary columns missing from raw data: {len(missing_dictionary_columns)}")
if missing_dictionary_columns:
    print(missing_dictionary_columns)

print(f"Columns with unmapped values: {len(unmapped_values)}")
if unmapped_values:
    for col, values in unmapped_values.items():
        print(f"{col}: {values}")

data.head()


In [ ]:

# Encode remaining categorical columns deterministically without requiring scikit-learn.
# Columns already encoded by the dictionary keep their ordinal values.
encoding_dict = {}

for col in data.columns:
    if pd.api.types.is_numeric_dtype(data[col]):
        continue

    non_null_values = list(pd.Series(data[col].dropna().unique()).sort_values(key=lambda s: s.astype(str)))
    if non_null_values:
        encoding_dict[col] = {value: idx for idx, value in enumerate(non_null_values)}

print(f"Automatically encoded categorical columns: {len(encoding_dict)}")
encoding_dict


In [ ]:

new_data = data.copy()

for col, mapping in encoding_dict.items():
    new_data[col] = new_data[col].map(mapping)

for col in new_data.columns:
    converted = pd.to_numeric(new_data[col], errors="coerce")
    if new_data[col].isna().equals(converted.isna()):
        new_data[col] = converted

new_data.head()


In [ ]:

# Save a full encoded audit file before selecting the analysis variables.
new_data.to_csv("validataion_all_columns.csv", index=False)
print(f"Saved full encoded data: validataion_all_columns.csv, shape={new_data.shape}")


In [ ]:

# The repository's anonymized raw data has Progress/Finished/Duration removed.
# If Finished is available, keep only completed responses; otherwise preserve all anonymized rows.
if "Finished" in new_data.columns:
    finished_non_null = new_data["Finished"].dropna()
    if not finished_non_null.empty:
        new_data = new_data[new_data["Finished"] == 1].copy()
        print(f"Filtered to completed responses; shape={new_data.shape}")
    else:
        print("Finished column is empty in the shared anonymized data; preserving all rows.")

metadata_cols = [
    "ResponseId",
    "Progress",
    "Duration (in seconds)",
    "Duration (in seconds) (Duration (in seconds))",
    "Finished",
    "RecordedDate",
]
new_data = new_data.drop(columns=metadata_cols, errors="ignore")
new_data.head()


In [ ]:

columns = [
    'background_time', 'background_work', 'clean_sat_2', 'background_hours',
    'light_sat_2', 'furnish_sat_3', 'amenities_sat_5', 'light_task',
    'amenities_sat_4', 'background_gender', 'thermal_sat_2', 'clean_sat_1',
    'iaq_sat_1', 'acoustic_sat_1', 'layout_sat_3', 'acoustic_sat_2',
    'thermal_sat_1', 'light_sat_4', 'light_sat_1', 'light_sat_3',
    'amenities_sat_3', 'support_sat_2', 'support_sat_1', 'layout_sat_2',
    'furnish_sat_1', 'furnish_sat_2', 'location_type', 'support_sat_3'
]

missing_analysis_columns = [col for col in columns if col not in new_data.columns]
if missing_analysis_columns:
    raise KeyError(f"Missing required analysis columns: {missing_analysis_columns}")

new_data = new_data[columns].copy()
print(f"Selected analysis data shape: {new_data.shape}")
new_data.head()


In [ ]:

# Optional causal discovery preview.
# Keep this disabled for reproducible preprocessing because it requires optional packages
# (cdt, pygraphviz) and a local R installation. The analysis notebook handles modelling.
RUN_CAUSAL_DISCOVERY = False
causal_graph = None
agraph = None

if RUN_CAUSAL_DISCOVERY:
    try:
        import cdt
        import networkx as nx
        from cdt.causality.graph import GES
        from IPython.display import Image

        df_skeleton = new_data.dropna().copy()
        output = GES().predict(df_skeleton)
        causal_graph = output

        try:
            agraph = nx.nx_agraph.to_agraph(output)
            agraph.layout(prog="dot")
            agraph.draw("file.png")
            display(Image("file.png"))
        except ImportError as exc:
            print(f"Graph image skipped because pygraphviz is unavailable: {exc}")
    except ImportError as exc:
        print(f"Causal discovery skipped because an optional dependency is unavailable: {exc}")
else:
    print("Skipping optional causal discovery in preprocessing notebook.")


In [ ]:

if agraph is not None:
    display(agraph.nodes())
else:
    print("No optional causal graph was generated.")


In [ ]:

if agraph is not None:
    display(agraph.edges())
else:
    print("No optional causal graph was generated.")


In [ ]:

if agraph is not None:
    import pygraphviz as pgv

    G = pgv.AGraph()
    for n in agraph.nodes():
        G.add_node(n)
    for e in agraph.edges():
        G.add_edge(e[0], e[1])
    print(G.string().replace("\n", " ").replace("\t", " ").replace("--", "->"))
else:
    print("No optional causal graph was generated.")


In [ ]:

# Keep the original misspelled filename for compatibility with 1_Analysis.ipynb.
new_data.to_csv("validataion.csv", index=False)
# Also write a correctly spelled alias for new users.
new_data.to_csv("validation.csv", index=False)

print(f"Saved validataion.csv and validation.csv; shape={new_data.shape}")
